# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

In [1]:
from pathlib import Path
from time import perf_counter

import tsplib95
import numpy as np

from wsp import ds, tsp

import sys

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 1.01
#ax = np.array([None, None])
#sys.setrecursionlimit(500_000)

In [2]:
#all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
#        continue

#    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
#    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
#    all_problems[problem.name] = ts_problem
#    print(f"Added {problem.name}")

#print("Found", len(all_problems), "euclidean TSPs")

In [3]:
#for name, problem in all_problems.items():
#    tour, _, _ = problem.nnn_path
#    badCount = 0
#    for (A, B), _ in problem.wspd:
#        pointsA = set(A.covered_points)
#        pointsB = set(B.covered_points)
#        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
#            badCount += 1
#    if badCount > 0:
#        print(f"Problem {name} has {badCount} bad pairs")

## Faster Runner using C libs

In [4]:
%load_ext line_profiler
from numba import njit
import elkai
from wspd import build_wspd, point as wsp_point # uses diam based sep factor

SIZE_THRESHOLD2 = 250

In [5]:
all_problems : list[tsplib95.models.StandardProblem] = []

for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
    if file.suffix != ".tsp" or not file.is_file():
        continue
    problem = tsplib95.load(f"{file.absolute()}")
    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
        continue
    if problem.dimension > SIZE_THRESHOLD2: # Skip large TSPs
        continue
    all_problems.append(problem)

In [ ]:
@njit(cache=True)
def wsp_heuristic_good(tour: np.ndarray, pos_in_tour: np.ndarray, A: np.ndarray, B: np.ndarray, state_arr: np.ndarray) -> bool:
    """A heuristic to check if a path is good based on the WSPs, checks if there are at least 2 connections between A and B"""
    assert len(A) > 0 and len(B) > 0, "Sets must be non-empty"

    exit_A, exit_B, biconn_count = 0, 0, 0
    num_points = len(pos_in_tour)

    state_arr[A] = 1
    state_arr[B] = 2

    for node in A:
        pos = pos_in_tour[node]

        v = state_arr[tour[pos + 1]] # since n+1 \in tour indices
        if v == 0: 
            exit_A += 1
        elif v == 2: 
            biconn_count += 1

        left_idx = num_points - 1 if pos == 0 else pos - 1
        u = state_arr[tour[left_idx]]
        if u == 0: 
            exit_A += 1

    for node in B:
        pos = pos_in_tour[node]

        v = state_arr[tour[pos + 1]]
        if v == 0: 
            exit_B += 1
        elif v == 1: 
            biconn_count += 1

        left_idx = num_points - 1 if pos == 0 else pos - 1
        u = state_arr[tour[left_idx]]
        if u == 0: 
            exit_B += 1 

    state_arr[A] = 0
    state_arr[B] = 0

    # single ham path case
    #if exit_A == 1 and exit_B == 1: # covered in the multi case
    #    return biconn_count == 1
    #if (exit_A == 2 and exit_B == 0) or (exit_A == 0 and exit_B == 2): # covered in the multi case
    #    return biconn_count == 2
    # mult ham path case
    if exit_A == 0 or exit_B == 0:
        return biconn_count == 2
    elif exit_A % 2 == 0:
        return biconn_count == 0
    else:
        return biconn_count == 1

In [12]:
def profile_bottleneck(problems):
    now = perf_counter()
    for prob in problems:
        print(f"Checking {prob.name}...")
        points = [wsp_point(prob.node_coords[i]) for i in prob.get_nodes()]
        wpsd: list[tuple[list[int], list[int]]] = []
        wspd = build_wspd(len(points), 2, S_FACTOR, points)

        # pruning problems which cannot be bad
        wspd = [(np.array(A, dtype=np.uint16), np.array(B, dtype=np.uint16)) for A, B in wspd if len(A) > 1 and len(B) > 1]

        lkh_instance = elkai.Coordinates2D({
            i-1: prob.node_coords[i] for i in prob.get_nodes()
        })

        # tour numpy conversions
        tour_lkh = lkh_instance.solve_tsp(runs=1)
        tour = np.array(tour_lkh, dtype=np.uint16)
        # vectorized node -> position map
        pos_in_tour = np.empty(len(tour) - 1, dtype=np.uint16)
        pos_in_tour[tour[:-1]] = np.arange(len(tour) - 1, dtype=np.uint16)

        state_arr = np.zeros(len(tour), dtype=np.uint8)
        for A, B in wspd:
            if not wsp_heuristic_good(tour, pos_in_tour, A, B, state_arr):
                print(f"Problem {prob.name} has a bad pair: {A}, {B}")

    #print("Checked all problems in", perf_counter() - now, "seconds")

%lprun -f profile_bottleneck -u 1.0 profile_bottleneck(all_problems)

Checking berlin52...
Checking bier127...
Checking ch130...
Checking ch150...
Checking d198...
Checking eil101...
Checking eil51...
Checking eil76...
Checking kroA100...
Checking kroA150...
Checking kroA200...
Checking kroB100...
Checking kroB150...
Checking kroB200...
Checking kroC100...
Checking kroD100...
Checking kroE100...
Checking lin105...
Checking pr107...
Checking pr124...
Checking pr136...
Checking pr144...
Checking pr152...
Checking pr226...
Checking pr76...
Checking rat195...
Checking rat99...
Checking rd100...
Checking st70...
Checking ts225...
Checking tsp225...
Checking u159...


Timer unit: 1 s

Total time: 5.18898 s
File: /var/folders/_t/4_m8pb3d3xs763zf2ztl10k40000gn/T/ipykernel_21697/986268421.py
Function: profile_bottleneck at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def profile_bottleneck(problems):
     2         1          0.0      0.0      0.0      now = perf_counter()
     3        33          0.0      0.0      0.0      for prob in problems:
     4        32          0.0      0.0      0.1          print(f"Checking {prob.name}...")
     5        32          0.0      0.0      0.2          points = [wsp_point(prob.node_coords[i]) for i in prob.get_nodes()]
     6        32          0.0      0.0      0.0          wpsd: list[tuple[list[int], list[int]]] = []
     7        32          0.1      0.0      1.4          wspd = build_wspd(len(points), 2, S_FACTOR, points)
     8                                           
     9                                                   # prunin